In [2]:
import av
from pathlib import Path
import os

In [13]:
example_dir = Path("~/Desktop/study/universal_manipulation_interface/example_demo_session").expanduser()
my_dir = Path("~/Documents").expanduser()

# os.listdir(my_dir)
# os.listdir(example_dir / "raw_videos" / "mapping.mp4")

In [ ]:
example_video = example_dir / "raw_videos" / "mapping.mp4"
my_video = my_dir / "mapping.bag"

my_dir.stat().st_ctime

from datetime import datetime
datetime.fromtimestamp(my_dir.stat().st_ctime)
# with av.open(str(example_video)) as container:
#     stream = container.streams.video[0]
#     print(stream.metadata)

# with av.open(str(my_video)) as container:
#     stream = container.streams.video[0]
#     print(stream.metadata)



datetime.datetime(2025, 11, 3, 17, 2, 28, 860339)

In [ ]:
import rospy
from sensor_msgs.msg import Image
from cv_bridge import CvBridge
import numpy as np

# global 변수로 플래그와 데이터 저장 공간을 만듭니다.
# ⚠️ 주의: Notebook 환경이 아닌 단일 스크립트 실행을 가정합니다.
global depth_frame_received
global depth_data
depth_frame_received = False
depth_data = None
bridge = CvBridge()
topic_name = "/camera/depth/image_raw"

def depth_callback(msg):
    """깊이 메시지를 수신하면 호출되는 콜백 함수"""
    global depth_frame_received
    global depth_data
    
    if not depth_frame_received:
        print("--- 📥 깊이 메시지 수신 완료 ---")
        
        try:
            # ROS Image 메시지를 NumPy 배열로 변환 (16비트 unsigned integer 형태 유지)
            # encoding="passthrough"를 사용하여 원본 인코딩(16UC1)을 그대로 사용합니다.
            cv_image = bridge.imgmsg_to_cv2(msg, desired_encoding="passthrough")
            
            # 1. 데이터 타입, 크기, 인코딩 확인
            print(f"토픽: {topic_name}")
            print(f"ROS 인코딩: {msg.encoding}") # 예: 16UC1
            print(f"NumPy Dtype: {cv_image.dtype}") # 예: uint16
            print(f"이미지 크기 (H x W): {cv_image.shape}") # 예: (480, 640)
            
            # 2. 데이터 값 출력 (일부만)
            # 배열의 중앙 부분 (예: 5x5)을 출력하여 값을 확인합니다.
            H, W = cv_image.shape
            center_h, center_w = H // 2, W // 2
            
            # 중앙 5x5 영역 추출
            center_patch = cv_image[center_h-2:center_h+3, center_w-2:center_w+3]
            
            print("\n--- 🔍 중앙 5x5 픽셀 값 (단위: mm) ---")
            print(center_patch)
            
            # 3. 최소/최대 값 확인
            print(f"\n최소 깊이 (Min Value): {np.min(cv_image)} mm")
            print(f"최대 깊이 (Max Value): {np.max(cv_image)} mm")
            
            # 4. 플래그 설정
            depth_data = cv_image
            depth_frame_received = True

        except Exception as e:
            print(f"CvBridge 변환 오류: {e}")
            
def run_one_frame_check():
    """메인 실행 함수"""
    global depth_frame_received
    
    # 1. ROS 노드 초기화
    rospy.init_node('single_frame_checker', anonymous=True)
    
    # 2. Subscriber 생성
    # queue_size=1을 사용하여 최신 메시지만 받습니다.
    rospy.Subscriber(topic_name, Image, depth_callback, queue_size=1)
    
    # 3. 루프 시작: 메시지를 받을 때까지 대기
    print(f"💡 토픽 **{topic_name}**으로부터 메시지를 기다리는 중...")
    
    rate = rospy.Rate(10) # 10Hz
    while not rospy.is_shutdown() and not depth_frame_received:
        rate.sleep()

    # 4. 메시지를 받았거나 ROS가 종료되면 루프 탈출
    if depth_frame_received:
        print("\n✅ 단일 프레임 처리를 완료하고 종료합니다.")
        # 노드를 명시적으로 종료 (rospy.spin()을 사용하지 않았기 때문에 필요)
        rospy.signal_shutdown("Frame received and processed.")
    else:
        print("\n❌ ROS가 종료되어 작업을 완료하지 못했습니다.")

# --- 실행 ---
if __name__ == '__main__':
    try:
        run_one_frame_check()
    except rospy.ROSInterruptException:
        pass

In [ ]:
depth_recorder_loop()